# Questão 1
## 1. Motivação
Como evoluiu a proporção de candidaturas femininas desde a obrigatoriedade do preenchimento da política de cotas?

A política de cotas de gênero nas candidaturas brasileiras começou em 1995, quando a Lei nº 9.100 previu que, no mínimo, 20% das vagas de cada partido ou coligação fossem preenchidas por candidaturas de mulheres nas eleições municipais de 1996. Em 1997, a Lei nº 9.504 (Lei das Eleições) passou a valer a partir das eleições de 1998 e elevou o patamar para o mínimo de 30% e o máximo de 70% de candidaturas de cada sexo, regra que alcança as eleições proporcionais: deputados federais, estaduais e distritais e vereadores. A redação original, porém, dizia que partidos e coligações deveriam “reservar” esses percentuais, o que deixava dúvida sobre o caráter obrigatório do preenchimento. Só em 2009 a Lei nº 12.034 trocou “deverá reservar” por “preencherá”, dando caráter impositivo à cota de 30%. Desde então, a cota funciona como uma condição coletiva de elegibilidade, verificada sobre o demonstrativo da lista de candidatos de cada partido. Como as eleições de 2010 foram as primeiras realizadas depois da lei de 2009, elas servem de marco para acompanharmos, na série de 2002 a 2024, como evoluiu a proporção de candidaturas femininas.

## 2. Hipótese
Supomos que a proporção de candidatas cresce a partir de 2010, quando o preenchimento passou a ser obrigatório, e se aproxima do mínimo de 30% nos cargos proporcionais, mas não nos majoritários, que não têm lista.

## 3. Análise dos dados

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

cols = ["ano", "tipo_eleicao", "cargo", "genero", "situacao"] # colunas de interesse para a questão
df = pd.read_parquet("../data/candidatos.parquet", columns=cols)
df["cargo_plot"] = df["cargo"].replace(
    {"deputado distrital": "deputado estadual/distrital",
     "deputado estadual": "deputado estadual/distrital"}
)
df.shape

(2925021, 6)

In [2]:
df.isna().sum()

ano                 0
tipo_eleicao        0
cargo               0
genero           2014
situacao        13076
cargo_plot          0
dtype: int64

In [3]:
for c in ["tipo_eleicao", "cargo", "genero", "situacao"]:
    print(df[c].value_counts(dropna=False).head(10), "\n")

print(sorted(df["ano"].unique()))
pd.crosstab(df["ano"], df["cargo"])   # quais cargos existem em quais anos

tipo_eleicao
eleicao ordinaria                        2920755
eleicoes suplementares 2008                  619
eleicoes municipais 2020 - ap                549
senador                                       39
eleicao suplementar senador - mt              35
eleicao suplementar - alto santo/ce           33
eleicao suplementar de boqueirao/pb           32
eleicoes suplementares tacaimbo 2023          26
eleicao suplementar de boa ventura/pb         20
eleicao suplementar monte horebe              20
Name: count, dtype: int64 

cargo
vereador              2578787
vice-prefeito          102076
prefeito               100492
deputado estadual       89572
deputado federal        41682
deputado distrital       4804
senador                  1598
vice-governador          1234
governador               1184
2º suplente               901
Name: count, dtype: int64 

genero
masculino    2064102
feminino      858905
None            2014
Name: count, dtype: int64 

situacao
deferido                  27

cargo,1º suplente,1º suplente senador,2º suplente,2º suplente senador,deputado distrital,deputado estadual,deputado federal,governador,prefeito,presidente,senador,vereador,vice-governador,vice-prefeito,vice-presidente
ano,,,,,,,,,,,,,,,
2002,0,313,0,312,663,11628,4356,197,0,6,313,0,196,0,6
2004,0,0,0,0,0,0,0,0,16238,0,0,368105,0,16513,0
2006,0,230,0,233,645,12196,5251,202,0,8,216,0,214,0,8
2008,0,0,0,0,0,0,0,0,15890,0,0,349252,0,16108,0
2010,0,295,0,307,884,14265,5940,168,0,9,268,0,178,0,9
2012,0,0,0,0,0,0,0,0,16114,0,0,449270,0,16467,0
2014,209,0,213,0,1025,16953,7074,191,0,12,182,0,199,0,11
2016,0,0,0,0,0,0,0,0,17124,0,0,462722,0,17418,0
2018,406,0,419,0,981,17916,8567,203,0,14,380,0,210,0,13


## 4. Decisões sobre a análise
Como a política de cotas vale para listas de candidatos, analisamos separadamente os cargos que possuem lista (proporcionais) e os que não possuem (majoritários). A análise foi feita sobre eleições ordinárias e foi dividida entre eleições gerais e municipais. 
Os cargos de vices e suplentes concorrem na chapa do titular e não compõem a lista que a cota regula, além de terem rótulos inconsistentes na base; por isso foram excluídos.
Deputado distrital foi agregado a deputado estadual, por serem cargos equivalentes (legislativo de unidade da federação, eleito por sistema proporcional e sujeito à cota). O DF tem 24 vagas, de modo que o cargo isolado produz proporções instáveis. A agregação altera em menos de 0,4 ponto percentual as proporções de candidatas e eleitas de deputado estadual em todas as eleições analisadas.
As 2014 candidaturas sem gênero foram excluídas, pois representavam um percentual pequeno dos dados.

### 4.1 Dados de 'situacao'

In [4]:
df[df["situacao"].isna()].groupby("ano").size()

ano
2024    13076
dtype: int64

In [5]:
d24 = df[df["ano"] == 2024]
print("Proporção de situação nula em 2024:", d24["situacao"].isna().mean() * 100, "\n")

nulos = d24[d24["situacao"].isna()]
print("Número de nulos por cargo:\n", nulos["cargo"].value_counts(), "\n")
print("Proporção de gênero entre os nulos:\n", nulos["genero"].value_counts(normalize=True), "\n")
print("Proporção de gênero entre o total:\n", d24["genero"].value_counts(normalize=True), "\n")

Proporção de situação nula em 2024: 2.8252823445781416 

Número de nulos por cargo:
 cargo
vereador         12209
vice-prefeito      505
prefeito           362
Name: count, dtype: int64 

Proporção de gênero entre os nulos:
 genero
masculino    0.630832
feminino     0.369168
Name: proportion, dtype: float64 

Proporção de gênero entre o total:
 genero
masculino    0.656853
feminino     0.343147
Name: proportion, dtype: float64 



As candidaturas com situação não preenchida na base são todas do ano de 2024, sendo aproximadamente 2,8% do total de candidaturas do ano. Como a proporção feminina entre elas é próxima à do restante da eleição, mantê-las ou excluí-las altera a proporção de candidatas em menos de 0,1 ponto percentual, então as mantivemos.

### 4.2 Definição do total de candidaturas

Para decidir se consideraríamos candidaturas inválidas no total, comparamos duas versões: 
- Versão 1, todas as registradas: conta toda candidatura que aparece na base, mesmo as que depois foram indeferidas, renunciaram ou foram canceladas.
- Versão 2, só as válidas: tira as indeferidas, renunciadas, canceladas etc., e conta só as que seguiram. 

Para a maioria das combinações de ano e cargo (39 de 48), a diferença entre as duas versões foi inferior a 1 ponto percentual. As exceções são os cargos majoritários com poucas candidaturas (presidente, senador), em que poucos registros alteram o percentual, e as eleições de 2010 e 2014 para deputado federal e estadual.
Em 2010 e 2014, as mulheres respondem por 35,1% e 48,5% dos registros indeferidos de deputado federal e estadual, contra cerca de 23% e 32% das candidaturas totais nesses cargos. Por isso, a definição do total altera a proporção feminina em 2 a 3 pontos percentuais nesses anos, então optamos por analisar ambas as versões. A base não informa o motivo do indeferimento, de modo que não é possível distinguir suas causas; retomamos o ponto na análise da pergunta 5.

In [6]:
base = df.copy()
# base = base[base["genero"].isin(["feminino", "masculino"])]

proporcionais = ["vereador", "deputado estadual", "deputado distrital", "deputado federal"]
majoritarios  = ["prefeito", "governador", "senador", "presidente"]
base = base[base["cargo"].isin(proporcionais + majoritarios)]

base["tipo_cargo"] = base["cargo"].map(lambda c: "Proporcional" if c in proporcionais else "Majoritário")
base["eleicao"]    = base["cargo"].map(lambda c: "Municipais" if c in ["vereador", "prefeito"] else "Gerais")

invalidas = ["indeferido", "indeferido com recurso", "renuncia", "cancelado",
             "cancelado com recurso", "cassado", "cassado com recurso",
             "falecido", "falecimento", "inelegivel", "impugnado",
             "nao conhecimento do pedido", "pedido nao conhecido",
             "pedido nao conhecido com recurso"]
base["valida"] = ~base["situacao"].isin(invalidas)

In [7]:
def pct_fem(d, chaves):
    t = d.groupby(chaves + ["genero"], observed=True).size().unstack("genero", fill_value=0)
    t["total"] = t["feminino"] + t["masculino"]
    t["pct_fem"] = t["feminino"] / t["total"] * 100
    return t.reset_index()

serie_todas   = pct_fem(base, ["ano", "cargo"])
serie_validas = pct_fem(base[base["valida"]], ["ano", "cargo"])

comp = serie_todas.merge(serie_validas, on=["ano", "cargo"], suffixes=("_todas", "_validas"))
comp["dif_pp"] = comp["pct_fem_validas"] - comp["pct_fem_todas"]
comp["dif_pp"].describe()

count    48.000000
mean      0.018776
std       1.344434
min      -3.067321
25%      -0.414632
50%      -0.003329
75%       0.149267
max       5.594406
Name: dif_pp, dtype: float64

In [8]:
cols = ["ano", "cargo", "total_todas", "total_validas",
        "pct_fem_todas", "pct_fem_validas", "dif_pp"]
comp.loc[comp["dif_pp"].abs() > 1, cols].sort_values("dif_pp")

genero,ano,cargo,total_todas,total_validas,pct_fem_todas,pct_fem_validas,dif_pp
18,2010,deputado federal,5940,4888,22.154882,19.087561,-3.067321
25,2014,deputado estadual,16945,14291,31.389790,29.053250,-2.336540
26,2014,deputado federal,7072,5868,31.674208,29.362645,-2.311563
17,2010,deputado estadual,14265,11811,22.754995,20.675641,-2.079353
36,2018,presidente,14,13,14.285714,15.384615,1.098901
29,2014,senador,182,164,18.681319,20.731707,2.050389
28,2014,presidente,12,11,25.000000,27.272727,2.272727
12,2006,presidente,8,7,25.000000,28.571429,3.571429
44,2022,presidente,13,11,30.769231,36.363636,5.594406


In [9]:
sub = base[base["ano"].isin([2010, 2014]) & base["cargo"].isin(["deputado federal", "deputado estadual/distrital"])]
inv = sub[~sub["valida"]]
tab = pd.crosstab([inv["ano"], inv["situacao"]], inv["genero"])
tab["total"] = tab.sum(axis=1)
tab["pct_fem"] = (tab["feminino"] / tab["total"] * 100).round(1)
tab.sort_values(["ano", "total"], ascending=[True, False])

genero                           feminino  masculino  total  pct_fem
ano  situacao                                                       
2010 indeferido                       311        487    798     39.0
     renuncia                          69        162    231     29.9
     indeferido com recurso             1         13     14      7.1
     nao conhecimento do pedido         1          4      5     20.0
     cancelado                          0          2      2      0.0
     cassado                            1          0      1    100.0
     falecido                           0          1      1      0.0
2014 indeferido                       397        438    835     47.5
     renuncia                         109        219    328     33.2
     nao conhecimento do pedido        10          8     18     55.6
     cancelado                          0         14     14      0.0
     indeferido com recurso             1          4      5     20.0
     falecido                           0          3      3      0.0
     cassado com recurso                0          1      1      0.0

Indeferimentos e renúncias somam 97% das candidaturas retiradas em cada ano, e o desvio feminino está no indeferimento (922 de 2.629 em 2010 e 1.246 de 2.569 em 2014).

## 5. Gráficos

In [10]:
long = comp.melt(id_vars=["ano", "cargo", "total_todas"],
                 value_vars=["pct_fem_todas", "pct_fem_validas"],
                 var_name="definicao", value_name="pct_fem")
long["definicao"] = long["definicao"].map({"pct_fem_todas": "Todas as registradas",
                                           "pct_fem_validas": "Só válidas"})
prop = long[long["cargo"].isin(proporcionais)].copy()
prop["eleicao"] = prop["cargo"].map(lambda c: "Municipais" if c == "vereador" else "Gerais")

In [11]:
prop = prop.copy()
prop["ano"] = prop["ano"].astype(int)
prop["cargo"] = prop["cargo"].astype(str)
prop["pct_fem"] = prop["pct_fem"].astype(float)
prop["total_todas"] = prop["total_todas"].astype(int)

print(prop.dtypes)
print(prop.shape, prop["pct_fem"].isna().sum())

ano              int64
cargo           object
total_todas      int64
definicao       object
pct_fem        float64
eleicao         object
dtype: object
(48, 6) 0


In [12]:
paleta = px.colors.qualitative.Safe
ordem = ["vereador", "deputado federal", "deputado estadual", "deputado distrital",
        "prefeito", "governador", "senador", "presidente"]
cores = {c: paleta[i] for i, c in enumerate(ordem)}

In [14]:
p1 = prop[prop["definicao"] == "Todas as registradas"]

fig1 = px.line(
    p1, x="ano", y="pct_fem", color="cargo", markers=True,
    facet_col="eleicao",
    category_orders={"eleicao": ["Gerais", "Municipais"]},
    hover_data={"total_todas": True},
    labels={"pct_fem": "% de candidaturas femininas", "ano": "Ano da eleição",
            "cargo": "Cargo", "total_todas": "Nº de candidaturas"},
    title="Percentual de candidaturas femininas para cargos proporcionais",
    color_discrete_map=cores
)
fig1.update_xaxes(matches=None)
fig1.update_xaxes(tickmode="linear", dtick=4, tick0=2002, col=1)
fig1.update_xaxes(tickmode="linear", dtick=4, tick0=2004, col=2)
fig1.update_yaxes(range=[0, 40], ticksuffix="%")
fig1.add_hline(y=30, line_dash="dot", annotation_text="Mínimo legal (30%)",
              annotation_position="bottom right", row="all", col="all")
fig1.add_vline(x=2010, line_dash="dash", line_color="gray", col=1)   # 1ª eleição geral após a Lei 12.034/2009
fig1.add_vline(x=2012, line_dash="dash", line_color="gray", col=2)   # 1ª eleição municipal após a lei
fig1.for_each_annotation(lambda a: a.update(text=a.text.replace("eleicao=", "")))
fig1.update_layout(height=500, legend_title_text="Cargo")
fig1.show()

In [15]:
p2 = prop[prop["cargo"].isin(["deputado federal", "deputado estadual"])]

fig2 = px.line(
    p2, x="ano", y="pct_fem", color="cargo", line_dash="definicao", markers=True,
    category_orders={"definicao": ["Todas as registradas", "Só válidas"]},
    hover_data={"total_todas": True},
    labels={"pct_fem": "% de candidaturas femininas", "ano": "Ano da eleição",
            "cargo": "Cargo", "definicao": "Total considerado",
            "total_todas": "Nº de candidaturas"},
    title="Em 2010 e 2014, excluir as candidaturas inválidas reduz a proporção feminina",
    color_discrete_map=cores
)
fig2.update_xaxes(tickmode="linear", dtick=4, tick0=2002)
fig2.update_yaxes(range=[10, 40], ticksuffix="%")
fig2.add_hline(y=30, line_dash="dot", annotation_text="Mínimo legal",
               annotation_position="bottom right")
fig2.add_vline(x=2010, line_dash="dash", line_color="gray")
fig2.add_annotation(
    text="Nota: o eixo vertical começa em 10%, para tornar visíveis diferenças pequenas.",
    xref="paper", yref="paper", x=0, y=-0.25,
    showarrow=False, xanchor="left", font=dict(size=11),
)
fig2.update_layout(margin=dict(b=110), height=500)
fig2.show()

In [16]:
maj = long[long["cargo"].isin(majoritarios) & (long["definicao"] == "Todas as registradas")].copy()
maj["eleicao"] = maj["cargo"].map(lambda c: "Municipais" if c == "prefeito" else "Gerais")
maj["ano"] = maj["ano"].astype(int)
maj["cargo"] = maj["cargo"].astype(str)
maj["pct_fem"] = maj["pct_fem"].astype(float)
maj["total_todas"] = maj["total_todas"].astype(int)

fig3 = px.line(
    maj, x="ano", y="pct_fem", color="cargo", markers=True,
    facet_col="eleicao",
    category_orders={"eleicao": ["Gerais", "Municipais"]},
    hover_data={"total_todas": True},
    labels={"pct_fem": "% de candidaturas femininas", "ano": "Ano da eleição",
            "cargo": "Cargo", "total_todas": "Nº de candidaturas"},
    title="Percentual de candidaturas femininas para cargos majoritários",
    color_discrete_map=cores
)
fig3.update_xaxes(matches=None)
fig3.update_xaxes(tickmode="linear", dtick=4, tick0=2002, col=1)
fig3.update_xaxes(tickmode="linear", dtick=4, tick0=2004, col=2)
fig3.update_yaxes(range=[0, 40], ticksuffix="%")
fig3.add_hline(y=30, line_dash="dot", annotation_text="Mínimo legal para cargos proporcionais",
              annotation_position="bottom right", row="all", col="all")
fig3.for_each_annotation(lambda a: a.update(text=a.text.replace("eleicao=", "")))
fig3.update_layout(height=500, legend_title_text="Cargo")
n_pres = maj.loc[maj["cargo"] == "presidente", "total_todas"]
texto = f"Nota: presidente tem entre {n_pres.min()} e {n_pres.max()} candidaturas por eleição, então o percentual oscila muito."
fig3.add_annotation(
    text=texto,
    xref="paper", yref="paper", x=0, y=-0.28,
    showarrow=False, xanchor="left", font=dict(size=11),
)
fig3.update_layout(margin=dict(b=120))
fig3.show()

## 6. Conclusão

Nos cargos proporcionais, a proporção de candidaturas femininas ficou bem abaixo de 30% em 2002 e 2006 nas eleições gerais (entre cerca de 11% e 21%, conforme o cargo) e em 2004 e 2008 nas municipais (cerca de 22%). Os maiores saltos ocorrem nas primeiras eleições posteriores à Lei nº 12.034/2009: em 2012, a proporção de candidatas a vereador sobe para cerca de 33%, e, nas eleições gerais, os cargos de deputado passam de cerca de 12–25% em 2006 para 22–25% em 2010 e ultrapassam 30% em 2014. A partir daí, a série cresce lentamente e permanece entre cerca de 31% e 35% até 2022 (gerais) e 2024 (municipais), isto é, próxima do mínimo legal e distante da participação feminina no eleitorado (52,8%).
Nos cargos majoritários, que não têm lista de candidatos e, portanto, não estão sujeitos à cota, a proporção de candidatas permanece bem abaixo de 30%. Prefeito, o cargo majoritário com mais candidaturas, passa de cerca de 9,5% em 2004 para cerca de 15% em 2024, enquanto vereador, nas mesmas eleições, vai de cerca de 22% a cerca de 35%. Governador e senador também crescem, de cerca de 10% e 12% em 2002 para cerca de 17% e 24% em 2022. O percentual de presidente oscila entre 0% e cerca de 31%, mas com poucas candidaturas por eleição, de modo que a variação não permite interpretação. A diferença de nível e o salto de 2012 apenas nos cargos com lista são compatíveis com um efeito da regra, mas a comparação é descritiva: os cargos majoritários diferem dos proporcionais em outros aspectos, e a alta lenta fora da cota indica que há outros fatores em jogo.

## 7. Limitações

Os dados analisados começam em 2002, mas a política de cotas de gênero começou em 1995; a base não informa o motivo dos indeferimentos de candidaturas; os nulos de situação em 2024; o n pequeno dos majoritários nacionais; e a figura 2 usa eixo truncado.